<a href="https://colab.research.google.com/github/isg-data/wanga_crypto_lab/blob/main/exp_02_bayesian_btc_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# Load libaries
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import pymc as pm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
# Set the start und end date
end_date = datetime(2025, 9, 30).strftime('%Y-%m-%d')
start_date = datetime(2015, 10, 1).strftime('%Y-%m-%d')

# Download Bitcoin data
btc = yf.download('BTC-USD', start=start_date, end=end_date, auto_adjust=True)

print(btc.head())
print(f"\nTotal records: {len(btc)}")

[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker         BTC-USD     BTC-USD     BTC-USD     BTC-USD   BTC-USD
Date                                                                
2015-10-01  237.548996  238.445007  235.615997  236.003998  20488800
2015-10-02  237.292999  238.541000  236.602997  237.264008  19677900
2015-10-03  238.729996  239.315002  236.944000  237.201996  16482700
2015-10-04  238.259003  238.968002  237.940002  238.531006  12999000
2015-10-05  240.382996  240.382996  237.035004  238.147003  23335900

Total records: 3652


In [5]:
# Create binary target variable
btc['Price_Up'] = (btc['High'] > btc['High'].shift(1)).astype(int)

# Create independent variables
btc['MA_3'] = btc['High'].rolling(window=3).mean()

btc['Volatility_3'] = btc['High'].rolling(window=3).std()

# RSI calculation
delta = btc['High'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=3).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=3).mean()
rs = gain / loss
btc['RSI_3'] = 100 - (100 / (1 + rs))

# Remove NaN values
btc = btc.dropna()

In [9]:
# Split data
train = btc[btc.index < pd.Timestamp(2022, 9, 30)]
test = btc[btc.index >= pd.Timestamp(2022, 10, 1)]

# Prepare training data
X_train = train[['MA_3', 'Volatility_3', 'RSI_3']].values
y_train = train['Price_Up'].values

# Standardize features
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0)
X_train_scaled = (X_train - X_mean) / X_std

In [10]:
# Bayesian logistic regression model
with pm.Model() as model:
    # Priors
    intercept = pm.Normal('intercept', mu=0, sigma=10)
    beta = pm.Normal('beta', mu=0, sigma=10, shape=3)

    # Logistic regression
    logit_p = intercept + pm.math.dot(X_train_scaled, beta)
    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))

    # Likelihood
    y_obs = pm.Bernoulli('y_obs', p=p, observed=y_train)

    # Sample
    trace = pm.sample(2000, tune=1000, return_inferencedata=True)

Output()

In [11]:
# Print summary
print(pm.summary(trace, var_names=['intercept', 'beta']))

            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
intercept -0.023  0.045  -0.106    0.060      0.001    0.001    3906.0   
beta[0]   -0.072  0.064  -0.189    0.048      0.001    0.001    2928.0   
beta[1]    0.110  0.067  -0.006    0.243      0.001    0.001    2708.0   
beta[2]    1.097  0.048   1.010    1.190      0.001    0.001    4110.0   

           ess_tail  r_hat  
intercept    3319.0    1.0  
beta[0]      2494.0    1.0  
beta[1]      2341.0    1.0  
beta[2]      3095.0    1.0  


In [12]:
# Make predictions on test set
X_test = test[['MA_3', 'Volatility_3', 'RSI_3']].values
X_test_scaled = (X_test - X_mean) / X_std
y_test = test['Price_Up'].values

posterior_means = trace.posterior.mean(dim=['chain', 'draw'])
intercept_mean = float(posterior_means['intercept'].values)
beta_mean = posterior_means['beta'].values

pred_probs = 1 / (1 + np.exp(-(intercept_mean + X_test_scaled @ beta_mean)))
predictions = (pred_probs > 0.5).astype(int)

In [18]:
# Evaluate
accuracy = (predictions == y_test).mean()
print(f"\nTest Accuracy: {accuracy:.3f}")


Test Accuracy: 0.670


In [19]:
# Fit logistic regression
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

# Make predictions
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

# Print results
print("Model Coefficients:")
print(f"Intercept: {log_reg.intercept_[0]:.4f}")
for i, feature in enumerate(['MA_3', 'Volatility_3', 'RSI_3']):
    print(f"{feature}: {log_reg.coef_[0][i]:.4f}")

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Coefficients:
Intercept: -1.5787
MA_3: -0.0000
Volatility_3: 0.0002
RSI_3: 0.0295

Test Accuracy: 0.671

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.72      0.69       559
           1       0.68      0.62      0.65       536

    accuracy                           0.67      1095
   macro avg       0.67      0.67      0.67      1095
weighted avg       0.67      0.67      0.67      1095


Confusion Matrix:
[[400 159]
 [201 335]]
